# GP_ELITE — dix minutes, quatre cellules

**Vous avez un tableau de mesures et vous voulez la formule, pas une prédiction.**
Ce notebook part de zéro et se termine sur *vos* données.

Rien à installer sur votre machine : tout tourne dans le navigateur.
Exécutez les cellules dans l'ordre avec **Maj + Entrée**.

---


## 1. Installation

Trois dépendances, aucun compilateur, aucun second langage.
Quelques secondes — c'est le point de départ de tout le reste.


In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'prêt.')


## 2. Votre première loi : Kepler, en huit points

Distance au Soleil et période orbitale des huit planètes. Rien d'autre.

Kepler a mis des années à établir cette relation. Voyons ce que la machine
en fait avec huit lignes de données.


In [ ]:
import numpy as np
from gp_elite import symbolic_regression

distance = np.array([0.387, 0.723, 1.000, 1.524, 5.203, 9.537, 19.191, 30.069])
periode  = np.array([0.241, 0.615, 1.000, 1.881, 11.862, 29.457, 84.011, 164.79])

resultat = symbolic_regression(
    distance.reshape(-1, 1), periode,
    feature_names=['distance'],
    operators='physical', generations=30, speed='fast', seed=0,
)
print(resultat.expression)


Vous devriez lire quelque chose comme `164.78 * (distance * sqrt(distance))`,
c'est-à-dire **distance^1.5** — la troisième loi de Kepler.

Le coefficient n'est qu'un facteur d'unités. C'est la *forme* qui est la loi.


## 3. Les unités physiques : ce que les autres ne font pas

Ici, une expérience de ressort. On mesure un allongement en mètres et une
force en newtons, et on cherche la loi de Hooke `F = k·x`.

La raideur `k` **n'est pas dans les données** — c'est une constante physique
inconnue, et elle porte des unités. On demande au moteur de la déduire.


In [ ]:
from gp_elite import GPEliteRegressor

rng = np.random.RandomState(0)
allongement = rng.uniform(0.01, 0.10, 150).reshape(-1, 1)   # mètres
force = 250.0 * allongement[:, 0]                            # newtons

est = GPEliteRegressor(
    units=['m'],              # unité de la colonne d'entrée
    target_units='N',         # unité de la cible
    unknown_constant=True,    # la constante manquante est à déduire
    generations=25, speed='fast', random_state=0,
)
est.fit(allongement, force)

print('unités déduites :', est.constant_units_string())
print('valeur déduite  :', round(est.constant_value_, 4))


`[kg / s^2]` et `250.0` — soit exactement des newtons par mètre, et la
raideur exacte du ressort.

Le moteur ne s'est pas contenté de trouver la forme de la loi : il a annoncé
**les unités et la valeur d'une grandeur physique absente des données**.

Sans `units=`, ce problème est hors d'atteinte — aucune constante sans
dimension ne peut relier des mètres à des newtons.


## 4. À votre tour

Chargez un CSV : une colonne cible, une ou plusieurs colonnes d'entrée.
Cent à cinq mille lignes, jusqu'à une dizaine de variables.


In [ ]:
import pandas as pd, io

try:
    from google.colab import files
    envoi = files.upload()
    nom = list(envoi)[0]
    df = pd.read_csv(io.BytesIO(envoi[nom]))
except ImportError:
    df = pd.read_csv('mes_donnees.csv')   # hors Colab : votre chemin ici

print(df.shape, 'lignes x colonnes')
df.head()


In [ ]:
# Adaptez ces deux lignes à vos colonnes.
CIBLE    = df.columns[-1]
ENTREES  = [c for c in df.columns if c != CIBLE]

X = df[ENTREES].to_numpy(dtype=float)
y = df[CIBLE].to_numpy(dtype=float)

res = symbolic_regression(X, y, feature_names=ENTREES,
                          operators='physical', generations=40,
                          speed='fast', seed=0)
print(res.expression)
print('R² sur données jamais vues :', res.r2_validation)


---

## Ça n'a pas marché ?

GP_ELITE est réglé sur des benchmarks publiés — propres, sans bruit, bien
échelonnés. Vos mesures ne sont rien de tout cela, et c'est précisément là
que le moteur a besoin de progresser.

Un échec sur vos données est une **information utile, pas une erreur de votre
part**. [Ouvrez une issue](https://github.com/ariel95500-create/gp-elite/issues/new/choose)
avec la forme de vos données et ce que vous avez obtenu — pas besoin de
partager les données, pas besoin de savoir pourquoi. Français ou anglais.

## Pour aller plus loin

- [README](https://github.com/ariel95500-create/gp-elite#readme) — mode robuste,
  front de Pareto, extrapolation, audit dimensionnel
- `gp-elite` en ligne de commande : une interface console sans écrire de Python
- [CHANGELOG](https://github.com/ariel95500-create/gp-elite/blob/main/CHANGELOG.md)
  — chaque affirmation du README est adossée à une mesure reproductible
